# DataMend STNet on Kaggle: SESSION A = main run, the architecture choice and (last, deliberate step) the test pass

Trains the main STNet run (seed 42), draws the per-kind ceiling check and the out-of-sample false-alarm check, fits the frozen STNet fusion with the rules and **makes the STNet choice** by the rule in `docs/qc-evaluation-protocol.md` (written once, `stnet_choice.json`). The old daily components are NOT retrained: the reference of the rule is read from the committed `ablation_s42.json` (same benchmark hash, checked). Set `DAILY = True` only if you want to retrain them. SYNTHETIC faults only.

The last cell runs the budgets and the **single-use test pass**. It does nothing until you set `CONFIRM_COMMITTED = True`, which you do only after downloading the zip, committing `stnet_choice.json` and pushing it.

Sessions: **A main** (this) and B hold (held-out fault kinds) run; C (no memory) and D (second seed) were skipped. Same STNet schedule in A and B so the results are comparable.

**How to run**
1. Kaggle: New Notebook, File > Import Notebook, upload this file.
2. Settings: **Accelerator** = GPU, **Internet** = On (needs a phone-verified account).
3. **Add Input**: the RF25 dataset `nathashalewis/majorproj-rainfall` (public).
4. If the GitHub repository is private: Add-ons > Secrets > add `GITHUB_TOKEN` (a token with read access) and switch it on for this notebook. Never paste a token into a cell.
5. First set `SMOKE = True` in the next cell and Run All: a short check (a few minutes) of everything except the test pass. Then set `SMOKE = False`, and use **Save Version > Save & Run All (Commit)** so it runs in the background, or keep the session open and let it run.
6. When the main cells finish, download `stnet_results_main.zip`, unzip it inside `ml-service/` of your checkout, commit and push. The choice is `artifacts/qc/base/stnet/stnet_choice.json`.
7. Then, in the SAME session (the checkpoint and scores live in its working folder), set `CONFIRM_COMMITTED = True` in the last cell and run it. Download `stnet_test_results.zip`, unzip it inside `ml-service/`, copy the new budget row into `docs/qc-evaluation-protocol.md` as the cell says, commit and push.

If a job stops at the time cap: Save Version, attach that output as an Input, put its link in `RESUME_LINK`, run again. It resumes from the last year-step.

Kaggle's terms allow one account per person: run each session from the account of the person running it.

In [ ]:
# ------------------------------- edit these -------------------------------
RUN_JOBS = ['main']          # this session's job
SMOKE = False                # True: a short check first (1 epoch x 5 steps, run name 'smoke'); then False for the real run
DAILY = False                # True: ALSO train the old daily components (spatial, temporal, plain GRU) and their fusion, and make the STNet choice here (needs them as the reference). Off by owner request: only the new STNet runs.
DATASET_LINK = '/kaggle/input/datasets/nathashalewis/majorproj-rainfall'
REPO_URL = 'https://github.com/NL0711/datamend.git'
BRANCH = 'main'
RUN_NAME = 'smoke' if SMOKE else 'base'
EPOCHS, STEPS, LONG_STEPS = (1, 5, 3) if SMOKE else (4, 200, 30)      # the real schedule is the same for every session
TIME_CAP_HOURS = 8.0         # per job; a job that reaches it stops cleanly and can be resumed
RESUME_LINK = ''             # link of a previous version's output (attached as input) to continue from
PUSH_RESULTS = False         # True: also commit the small results to branch kaggle-results-<jobs> (needs a GITHUB_TOKEN secret with write access)
EXPECTED_BENCH_HASH = 'c37de7e3cf2be90e'    # the frozen benchmark every result so far was scored on; a different hash stops the run
JOBS = {
    'main':     dict(prefix='stnet',          seed=42, args=[], post=True),
    'seed43':   dict(prefix='stnet',          seed=43, args=[], post=False),
    'hold':     dict(prefix='stnet_hold',     seed=42, args=['--holdout-kinds', 'zeroed', 'scale', 'drift'], post=False),
    'nomem':    dict(prefix='stnet_nomem',    seed=42, args=['--no-gru', '--no-long'], post=False),
    'norobust': dict(prefix='stnet_norobust', seed=42, args=['--no-robust'], post=False),
    'nodisc':   dict(prefix='stnet_nodisc',   seed=42, args=['--no-disc'], post=False),
}
unknown = [j for j in RUN_JOBS if j not in JOBS]
if unknown: raise ValueError('unknown job(s): ' + ', '.join(unknown) + '. Choose from ' + ', '.join(JOBS))
print('session job(s):', RUN_JOBS, '| old daily components:', DAILY, '| run name:', RUN_NAME, '| schedule:', EPOCHS, 'epochs x', STEPS, 'steps', '(SMOKE: short check)' if SMOKE else '')

In [ ]:
import os, sys, subprocess, pathlib, shutil, json, time, importlib
WORK = pathlib.Path('/kaggle/working') if pathlib.Path('/kaggle/working').is_dir() else pathlib.Path.cwd()
SECRET = ''
try:
    from kaggle_secrets import UserSecretsClient
    SECRET = UserSecretsClient().get_secret('GITHUB_TOKEN')
except Exception as e:
    print('No GITHUB_TOKEN secret (fine for a public repository):', type(e).__name__)
def clean(t): return t.replace(SECRET, '***') if SECRET else t
def sh(cmd, cwd=None, show=True):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    out = clean((r.stdout + r.stderr).strip())
    if show: print('$', clean(cmd), chr(10) + out[-1200:])
    if r.returncode: raise RuntimeError('failed: ' + clean(cmd)[:120] + chr(10) + out[-1200:])
    return out
def auth_url(url, token): return url.replace('https://', 'https://x-access-token:' + token + '@') if token else url
os.environ['GIT_TERMINAL_PROMPT'] = '0'
REPO = WORK / 'datamend'
url = auth_url(REPO_URL, SECRET)
try:
    if (REPO / '.git').is_dir():
        sh('git -C ' + str(REPO) + ' fetch ' + url + ' ' + BRANCH + ' && git -C ' + str(REPO) + ' checkout -B ' + BRANCH + ' FETCH_HEAD')
    else:
        sh('git clone --branch ' + BRANCH + ' ' + url + ' ' + str(REPO))
        sh('git -C ' + str(REPO) + ' remote set-url origin ' + REPO_URL, show=False)
except RuntimeError as e:
    raise RuntimeError('Could not get the repository. Check that Internet is ON in the notebook settings and, if the repository is private, that the GITHUB_TOKEN secret is attached to this notebook.') from None
sh('git -C ' + str(REPO) + ' log -1 --oneline')
ML = REPO / 'ml-service'
os.chdir(ML)
sys.path.insert(0, str(ML))
for _m in [k for k in sys.modules if k == 'qc' or k.startswith('qc.')]:
    del sys.modules[_m]
importlib.invalidate_caches()
sh('pip install -q numpy pandas scipy scikit-learn xarray netCDF4 matplotlib pyyaml pyarrow kagglehub psutil nvidia-ml-py', show=False)
print('repo:', REPO, '| cwd:', os.getcwd())

In [ ]:
import torch
from qc.kaggle_data import locate_rf25, find_saved_outputs
from qc.config import load_config, year_range

cfg0 = load_config(ML / 'qc' / 'configs' / 'base.yaml', [])
need = sum(len(year_range(cfg0.years[k])) for k in ('train', 'val', 'test', 'stress') if cfg0.years.get(k))
def _download(slug):
    import kagglehub
    return kagglehub.dataset_download(slug)
RF25_DIR = locate_rf25(DATASET_LINK, expected_files=need, download=_download)
n_gpu = torch.cuda.device_count()
print('RF25:', RF25_DIR, '|', len(list(RF25_DIR.glob('RF25_ind*_rfp25.nc'))), 'files | GPUs:', n_gpu, [torch.cuda.get_device_name(i) for i in range(n_gpu)])
if n_gpu == 0: raise RuntimeError('No GPU: turn on a GPU accelerator in the notebook settings.')
if len(RUN_JOBS) > n_gpu: raise RuntimeError('RUN_JOBS has ' + str(len(RUN_JOBS)) + ' jobs but there are ' + str(n_gpu) + ' GPU(s): one job per GPU.')
os.environ['DATAMEND_RF25_DIR'] = str(RF25_DIR)
os.environ['DATAMEND_DATA_DIR'] = str(WORK / 'data')
OUT_ROOT = WORK / 'artifacts'
RUN = RUN_NAME
BASE = ['--profile', 'single-gpu', '--set', 'name=' + RUN]

# Continue from a previous session (optional)
if RESUME_LINK.strip():
    prev = find_saved_outputs(RESUME_LINK)
    if prev is None:
        raise RuntimeError('no previous output (a folder with artifacts/qc) found for ' + RESUME_LINK + '. Attach the previous version output as an input first.')
    for sub in ('artifacts', 'data'):
        if (prev / sub).is_dir():
            shutil.copytree(prev / sub, WORK / sub, dirs_exist_ok=True)
    print('restored from', prev)
else:
    print('fresh run (no RESUME_LINK)')

In [ ]:
def run_stream(step, *args):
    """Run `python -m qc <step>`, stream its output, return it as text. Raises if the step fails."""
    cmd = [sys.executable, '-u', '-m', 'qc', step, *args, *BASE]
    print('$', ' '.join(cmd))
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=ML)
    lines = []
    for line in p.stdout:
        print(line, end='')
        lines.append(line)
    if p.wait():
        raise RuntimeError('step failed: ' + step)
    return ''.join(lines)

# Shared, cheap, identical on every account: data check and splits, the frozen fault benchmark, the untrained baselines (rules are needed for the fusion).
run_stream('prepare')
run_stream('benchmark')
run_stream('untrained')

from qc.benchmark import bench_dir
cfg = load_config(ML / 'qc' / 'configs' / 'base.yaml', ['name=' + RUN])
bench_hash = json.loads((bench_dir(cfg) / 'manifest.json').read_text())['hash']
print('benchmark hash:', bench_hash)
if bench_hash != EXPECTED_BENCH_HASH:
    raise RuntimeError('benchmark hash ' + bench_hash + ' is not the expected ' + EXPECTED_BENCH_HASH + ': results from this notebook would not be comparable with the earlier ones. Stop and check the repository version.')

In [ ]:
# One process per job, each on its own GPU. Logs go to /kaggle/working/job_<name>.log
# With two GPUs the old daily components (spatial, temporal, plain GRU) train on the spare one at the same time; with one GPU they train first.
DAILY_ARGS = ['--seeds', '42', '--time-cap-hours', '2.0'] + (['--set', 'train.epochs=1', '--set', 'train.steps_per_year=5', '--max-train-years', '3'] if SMOKE else [])
daily_proc, daily_log, daily_unfinished = None, WORK / 'job_daily.log', False
run_daily = DAILY and 'main' in RUN_JOBS
if run_daily:
    if n_gpu > len(RUN_JOBS):
        dcmd = [sys.executable, '-u', '-m', 'qc', 'train', *DAILY_ARGS, *BASE, '--set', 'compute.devices=[cuda:' + str(len(RUN_JOBS)) + ']']
        daily_proc = subprocess.Popen(dcmd, stdout=open(daily_log, 'a'), stderr=subprocess.STDOUT, text=True, cwd=ML)
        print('started the old daily components on cuda:' + str(len(RUN_JOBS)), 'at the same time as STNet')
    else:
        print('one GPU only: training the old daily components first, then STNet')
        daily_unfinished = 'stopped at the time cap' in run_stream('train', *DAILY_ARGS)

procs = []
for i, name in enumerate(RUN_JOBS):
    j = JOBS[name]
    cmd = [sys.executable, '-u', '-m', 'qc', 'stnet', '--seeds', str(j['seed']), '--prefix', j['prefix'], '--epochs', str(EPOCHS), '--steps', str(STEPS),
           '--long-steps', str(LONG_STEPS), '--time-cap-hours', str(TIME_CAP_HOURS), *j['args'], *BASE, '--set', 'compute.devices=[cuda:' + str(i) + ']']
    logf = open(WORK / ('job_' + name + '.log'), 'a')
    procs.append((name, subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT, text=True, cwd=ML), logf))
    print('started', name, 'on cuda:' + str(i), '|', ' '.join(cmd[4:]))

done = {}
while len(done) < len(procs):
    time.sleep(60)
    for name, p, logf in procs:
        if name in done:
            continue
        rc = p.poll()
        try:
            tail = (WORK / ('job_' + name + '.log')).read_text().strip().splitlines()[-1][:150]
        except Exception:
            tail = ''
        print(time.strftime('%H:%M'), name, 'running' if rc is None else 'finished rc=' + str(rc), '|', tail)
        if rc is not None:
            done[name] = rc
            logf.close()
bad = [n for n, rc in done.items() if rc]
if bad:
    raise RuntimeError('job(s) failed: ' + ', '.join(bad) + '. Read /kaggle/working/job_<name>.log (the last lines say what went wrong).')
unfinished = [n for n in RUN_JOBS if 'stopped at the time cap' in str((WORK / ('job_' + n + '.log')).read_text().splitlines()[-3:])]
if unfinished:
    print('STOPPED AT THE TIME CAP:', unfinished, '- Save Version, attach the output as an input, set RESUME_LINK and run again. The post steps below are skipped for these.')

if daily_proc is not None:
    print('waiting for the old daily components to finish ...')
    rc = daily_proc.wait()
    if rc:
        raise RuntimeError('the old daily components failed. Read /kaggle/working/job_daily.log (the last lines say what went wrong).')
    daily_unfinished = 'stopped at the time cap' in daily_log.read_text()[-3000:]
if daily_unfinished:
    print('THE OLD DAILY COMPONENTS STOPPED AT THE TIME CAP: Save Version, attach the output as an input, set RESUME_LINK and run again. The choice below is skipped.')

In [ ]:
# Fusion with the untrained rules, the per-kind ceiling check and the out-of-sample false-alarm check, for the main job only.
# Then the frozen STNet fusion and the STNet CHOICE (the rule in docs/qc-evaluation-protocol.md). The old daily fusion is the reference: read from
# the committed ablation file unless DAILY trained the old components here.
for name in RUN_JOBS:
    j = JOBS[name]
    if not j['post'] or name in unfinished:
        continue
    s, pre = str(j['seed']), j['prefix']
    run_stream('ablate', '--seed', s, '--tag', pre, '--subset', f'rules+{pre}_fault', '--subset', f'rules+{pre}_fault+{pre}_clean',
               '--subset', f'rules+{pre}_fault+{pre}_zeroed+{pre}_scale+{pre}_drift')
    run_stream('ceiling', '--seed', s, '--tag', pre, '--components', *[f'{pre}_{k}_s{s}' for k in ('fault', 'clean', 'disc', 'zeroed', 'scale', 'drift')])
    run_stream('calibrate', '--seed', s, '--components', f'{pre}_fault_s{s}')
    if run_daily and not daily_unfinished:
        run_stream('fusion', '--seeds', s)                      # the old daily fusion (fusion_logistic_s42, fusion_boosted_s42), used as the reference
    if not daily_unfinished:
        run_stream('stnet-fuse', '--seed', s, '--prefix', pre)  # frozen fusion of the rules and the STNet fault head, regional tables, validation curves
        choice_file = OUT_ROOT / 'qc' / RUN / 'stnet' / 'stnet_choice.json'
        if choice_file.is_file():
            print('the choice was already made in an earlier session and is written once:', choice_file)
        else:
            run_stream('choose-stnet', '--seed', s, '--prefix', pre,
                       '--reference-json', str(ML / 'artifacts' / 'qc' / 'base' / 'ablation' / ('ablation_s' + s + '.json')))
        choice = json.loads(choice_file.read_text())
        print('CHOICE:', choice['chosen'], '|', choice['reason'])
        print('The single-use test pass is the LAST cell: run it only after you have committed this choice.')

In [ ]:
# Package the small results: tables, json reports, curve images, telemetry. No checkpoints, no score arrays.
import zipfile
run_dir = OUT_ROOT / 'qc' / RUN
keep = []
for pat in ('**/*.md', '**/*.json', '**/*.png'):
    keep += [p for p in run_dir.glob(pat) if p.is_file() and p.stat().st_size < 50e6 and 'scores' not in p.parts and 'checkpoints' not in p.parts]
keep = list(dict.fromkeys(keep))
zname = WORK / ('stnet_results_' + '_'.join(RUN_JOBS) + ('_smoke' if SMOKE else '') + '.zip')
with zipfile.ZipFile(zname, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in keep:
        z.write(p, p.relative_to(WORK))
print(zname, '(' + format(zname.stat().st_size / 1e6, '.1f') + ' MB,', len(keep), 'files)')
if SMOKE:
    print('SMOKE check finished. If nothing failed above, set SMOKE = False and run the real thing.')
else:
    print('Unzip it inside ml-service/ of your checkout (it holds artifacts/qc/' + RUN + '/...), then commit and push.')

if PUSH_RESULTS and not SMOKE:
    if not SECRET:
        print('PUSH_RESULTS skipped: no GITHUB_TOKEN secret (it needs write access).')
    else:
        try:
            from qc.kaggle_sync import sync_results
            branch = 'kaggle-results-' + '-'.join(RUN_JOBS)
            files = {p: str(p.relative_to(WORK)) for p in keep}
            sha = sync_results(REPO, auth_url(REPO_URL, SECRET), RUN, files, branch, {}, message='results: STNet ' + ', '.join(RUN_JOBS), secret=SECRET)
            print('pushed to', branch, sha)
        except Exception as e:
            print('push failed (the zip is still there):', clean(str(e))[:300])

In [ ]:
# LAST STEP, run only AFTER you have downloaded the zip above, committed artifacts/qc/base/stnet/stnet_choice.json and pushed it:
# the budgets (logged in the protocol) and the single-use STNet test pass. The test years can be scored ONCE.
CONFIRM_COMMITTED = False      # True = "I committed and pushed stnet_choice.json; the test and stress years may be scored once"
BUDGET_FINAL, BUDGET_REVIEW = 1.0, 5.0     # false alerts per 1,000 clean cell-days (owner's choice, 2026-11-18); fixed once logged
if SMOKE or not CONFIRM_COMMITTED:
    print('skipped (SMOKE, or CONFIRM_COMMITTED is False). Nothing was scored on a test year.')
else:
    s = str(JOBS['main']['seed'])
    run_stream('set-budgets', '--seed', s, '--final', str(BUDGET_FINAL), '--review', str(BUDGET_REVIEW))
    run_stream('stnet-test')
    zname2 = WORK / 'stnet_test_results.zip'
    with zipfile.ZipFile(zname2, 'w', zipfile.ZIP_DEFLATED) as z:
        for p in list((OUT_ROOT / 'qc' / RUN).glob('**/*.md')) + list((OUT_ROOT / 'qc' / RUN).glob('**/*.json')) + list((OUT_ROOT / 'qc' / RUN).glob('**/*.png')):
            if p.is_file() and p.stat().st_size < 50e6 and 'scores' not in p.parts and 'checkpoints' not in p.parts:
                z.write(p, p.relative_to(WORK))
        z.write(REPO / 'docs' / 'qc-evaluation-protocol.md', 'protocol_with_budget_row.md')
    print(zname2, '- download it and unzip it inside ml-service/. Then open protocol_with_budget_row.md, copy ONLY the new last row of its budgets table into docs/qc-evaluation-protocol.md of your checkout (do not overwrite that file), delete protocol_with_budget_row.md, commit and push.')